In [1]:
# prompt: mount drive

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install wandb -qU
import wandb
wandb.login()


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sblas (sblas-universidad-nacional-del-litoral) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
sweep_config = {
    'method': 'bayes',  # Puede ser random, grid o bayes
    'metric': {
        'name': 'val_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'latent_dim': {
            'values': [128, 250, 512, 1024]
        },
        'hidden_dim': {
            'values': [1024, 2048, 3072, 4096, 5120, 6144]
        },
        'beta': {
            'distribution': 'uniform',
            'min': 0.01,
            'max': 5.0
        },
        'dropout': {
            'values': [0.0, 0.001, 0.01, 0.1]
        },
        'lr': {
            'distribution': 'log_uniform_values',
            'min': 1e-6,
            'max': 1e-3
        },
        'batch_size': {
            'values': [16, 32, 64]
        },
        'weight_decay': {
            'distribution': 'log_uniform_values',
            'min': 1e-8,
            'max': 1e-4
        }
    }
}

import pprint
pprint.pprint(sweep_config)


{'method': 'bayes',
 'metric': {'goal': 'minimize', 'name': 'val_loss'},
 'parameters': {'batch_size': {'values': [16, 32, 64]},
                'beta': {'distribution': 'uniform', 'max': 5.0, 'min': 0.01},
                'dropout': {'values': [0.0, 0.001, 0.01, 0.1]},
                'hidden_dim': {'values': [1024, 2048, 3072, 4096, 5120, 6144]},
                'latent_dim': {'values': [128, 250, 512, 1024]},
                'lr': {'distribution': 'log_uniform_values',
                       'max': 0.001,
                       'min': 1e-06},
                'weight_decay': {'distribution': 'log_uniform_values',
                                 'max': 0.0001,
                                 'min': 1e-08}}}


In [4]:
sweep_id = wandb.sweep(sweep_config, project="vae-hyperparam-search")


Create sweep with ID: dh5d9j0g
Sweep URL: https://wandb.ai/sblas-universidad-nacional-del-litoral/vae-hyperparam-search/sweeps/dh5d9j0g


In [5]:
# Función de pérdida
def loss_function(recon_x, x, mu, logvar, beta):
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + beta * kld
    return total_loss, recon_loss.item(), kld.item()

In [6]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import os
import sys
# Get the absolute path to the directory containing 'models'
# Assuming your 'models' directory is in '/content/drive/MyDrive/1st_paper/'
models_dir = os.path.join('/content/drive/MyDrive/1st_paper/')

# Append it to sys.path
sys.path.append(models_dir)

# The rest of your code from ipython-input-6-b9d3ab608c4b
from models.vae import BetaVAE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def train():
    # Inicializa W&B run
    wandb.init()

    config = wandb.config

    # Carga tus datos (modifica con tu path correcto)
    train_data = torch.load('/content/drive/MyDrive/1st_paper/fold_1/train_data_normed.pt')
    val_data = torch.load('/content/drive/MyDrive/1st_paper/fold_1/val_data_normed.pt')

    train_loader = DataLoader(TensorDataset(train_data), batch_size=config.batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(TensorDataset(val_data), batch_size=config.batch_size, num_workers=4)

    # Inicializa modelo con hiperparámetros del Sweep
    model = BetaVAE(config.latent_dim, config.hidden_dim, config.beta, config.dropout, input_channels=4).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

    best_val_loss = float('inf')

    epochs = 80  # Puedes ajustar esto o añadirlo al sweep_config

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x, in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            recon, mu, logvar, _ = model(x)
            loss, _, _ = loss_function(recon, x, mu, logvar, config.beta)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)

        train_loss /= len(train_loader.dataset)

        # Evalúa
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, in val_loader:
                x = x.to(device)
                recon, mu, logvar, _ = model(x)
                loss, _, _ = loss_function(recon, x, mu, logvar, config.beta)
                val_loss += loss.item() * x.size(0)

        val_loss /= len(val_loader.dataset)

        # Scheduler
        scheduler.step(val_loss)

        # Guarda modelo si es mejor
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'/content/drive/MyDrive/1st_paper/wandb_best_model.pth')

        # Loguear métricas en W&B
        wandb.log({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss
        })


Using device: cuda


In [7]:
wandb.agent(sweep_id, train, count=60)  # Realiza 20 experimentos diferentes


wandb: Agent Starting Run: pb4h8ryt with config:
wandb: 	batch_size: 64
wandb: 	beta: 4.114711971997744
wandb: 	dropout: 0.001
wandb: 	hidden_dim: 2048
wandb: 	latent_dim: 1024
wandb: 	lr: 1.9107047743200413e-05
wandb: 	weight_decay: 7.69200702725987e-05


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train_loss,█▅▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▁▁▄▅▆▇██████████████████████████████████
epoch,79
train_loss,6467.366
val_loss,9779.28418


wandb: Agent Starting Run: djifdh3o with config:
wandb: 	batch_size: 64
wandb: 	beta: 3.2829525811094693
wandb: 	dropout: 0
wandb: 	hidden_dim: 4096
wandb: 	latent_dim: 128
wandb: 	lr: 5.041668663955423e-06
wandb: 	weight_decay: 1.3441316098938746e-08


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇████
train_loss,█▆▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▁▁▂▃▆▇▇█████████████████████████████████
epoch,79
train_loss,500.603
val_loss,1524.93884


...

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train_loss,▇█▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,79
train_loss,51.20086
val_loss,3.88999


wandb: Agent Starting Run: rjcavkku with config:
wandb: 	batch_size: 32
wandb: 	beta: 3.5669203638252376
wandb: 	dropout: 0.1
wandb: 	hidden_dim: 4096
wandb: 	latent_dim: 128
wandb: 	lr: 0.0004359879266276726
wandb: 	weight_decay: 9.98868715833583e-08


epoch,▁▁▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
train_loss,█▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▂█▃▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,79
train_loss,185.50033
val_loss,3.58823


wandb: Agent Starting Run: 217fwdhh with config:
wandb: 	batch_size: 32
wandb: 	beta: 3.8930321180850807
wandb: 	dropout: 0.1
wandb: 	hidden_dim: 3072
wandb: 	latent_dim: 128
wandb: 	lr: 0.0005344553822379303
wandb: 	weight_decay: 2.762752268383314e-07


epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
train_loss,█▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,79
train_loss,196.52233
val_loss,2.51777
